<a href="https://colab.research.google.com/github/parthpatel182003/Sentinel-2/blob/main/notebooks/02_preprocess_phase1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 Preprocess And Phase 1 Image Processing

This notebook prepares the downloaded bands and builds the classical water detection baseline.


Steps:
1. Load Sentinel-2 TIFF images
2. Compute MNDWI index
3. Apply Otsu thresholding
4. Remove noise and artifacts
5. Remove ocean region
6. Save masks and overlays


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

REPO_PATH = "/content/WaterDetectionProjectRepo"

if not os.path.exists(REPO_PATH):
    !git clone https://github.com/parthpatel182003/Sentinel-2.git {REPO_PATH}

%cd {REPO_PATH}

Cloning into '/content/WaterDetectionProjectRepo'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 37 (delta 11), reused 19 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 3.76 MiB | 24.40 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/WaterDetectionProjectRepo


In [3]:
import numpy as np
import rasterio
import cv2
import os
import matplotlib.pyplot as plt
import pandas as pd

In [4]:
BASE_PATH = "/content/drive/MyDrive/WaterDetectionProject"

image_dir = BASE_PATH + "_exports"
mask_dir = BASE_PATH + "/masks"
overlay_dir = BASE_PATH + "/overlays"

os.makedirs(mask_dir, exist_ok=True)
os.makedirs(overlay_dir, exist_ok=True)

print("Paths ready ✅")

Paths ready ✅


In [5]:
def load_sentinel_image(path):
    with rasterio.open(path) as src:
        return {
            "blue": src.read(1),
            "green": src.read(2),
            "red": src.read(3),
            "nir": src.read(4),
            "swir": src.read(5),
            "meta": src.meta
        }


def compute_indices(bands):
    green = bands["green"].astype(np.float32)
    swir = bands["swir"].astype(np.float32)

    mndwi = (green - swir) / (green + swir + 1e-6)
    return mndwi


def otsu_threshold(image):
    image = np.nan_to_num(image)

    image_norm = (image - image.min()) / (image.max() - image.min() + 1e-6)
    image_8bit = (image_norm * 255).astype(np.uint8)

    _, thresh = cv2.threshold(
        image_8bit, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    kernel = np.ones((3, 3), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

    return thresh


def remove_small_components(mask):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)

    clean_mask = np.zeros_like(mask)

    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        width = stats[i, cv2.CC_STAT_WIDTH]
        height = stats[i, cv2.CC_STAT_HEIGHT]

        aspect_ratio = max(width, height) / (min(width, height) + 1e-6)

        if area > 300 and aspect_ratio < 5:
            clean_mask[labels == i] = 255

    return clean_mask


def remove_ocean_largest_component(mask):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)

    largest_label = 1
    largest_area = 0

    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] > largest_area:
            largest_area = stats[i, cv2.CC_STAT_AREA]
            largest_label = i

    mask[labels == largest_label] = 0

    return mask


def save_mask(mask, meta, output_path):
    meta.update({
        "count": 1,
        "dtype": "uint8"
    })

    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(mask, 1)


def create_overlay(red, green, blue, mask, output_path):
    rgb = np.stack([red, green, blue], axis=-1)
    rgb = (rgb / (np.max(rgb) + 1e-6) * 255).astype(np.uint8)

    overlay = rgb.copy()
    overlay[mask == 255] = [255, 0, 0]

    plt.imshow(overlay)
    plt.axis("off")
    plt.savefig(output_path, bbox_inches="tight", pad_inches=0)
    plt.close()

## To Implement

- read required bands for each scene
- align resolutions when needed
- compute NDWI and MNDWI
- apply thresholding
- refine masks with morphology
- save masks, overlays, and area summaries

In [6]:
def process_image(image_path):
    name = os.path.basename(image_path).replace(".tif", "")

    bands = load_sentinel_image(image_path)
    mndwi = compute_indices(bands)

    mask = otsu_threshold(mndwi)
    mask = remove_small_components(mask)
    mask = remove_ocean_largest_component(mask)

    mask_path = os.path.join(mask_dir, f"{name}_water_mask_mndwi.tif")
    overlay_path = os.path.join(overlay_dir, f"{name}_water_overlay.png")

    save_mask(mask, bands["meta"], mask_path)
    create_overlay(
        bands["red"], bands["green"], bands["blue"],
        mask, overlay_path
    )

    water_pixels = np.sum(mask == 255)
    total_pixels = mask.size

    return {
        "image": name,
        "water_pixels": int(water_pixels),
        "water_ratio": float(water_pixels / total_pixels)
    }

In [7]:
results = []

files = sorted([f for f in os.listdir(image_dir) if f.endswith(".tif")])

for idx, file in enumerate(files):
    print(f"[{idx+1}/{len(files)}] Processing: {file}")

    path = os.path.join(image_dir, file)
    result = process_image(path)
    results.append(result)

df = pd.DataFrame(results)

save_path = BASE_PATH + "/phase1_summary.csv"
df.to_csv(save_path, index=False)

print("\nSaved summary to:", save_path)
print("\nProcessing complete ✅")
print(f"Total images processed: {len(results)}")

df.head()

[1/14] Processing: kuttanad_kerala_scene_1.tif
[2/14] Processing: kuttanad_kerala_scene_2.tif
[3/14] Processing: kuttanad_kerala_scene_3.tif
[4/14] Processing: kuttanad_s2_first_image.tif
[5/14] Processing: s2_scene_0.tif
[6/14] Processing: s2_scene_1.tif
[7/14] Processing: s2_scene_2.tif
[8/14] Processing: s2_scene_3.tif
[9/14] Processing: s2_scene_4.tif
[10/14] Processing: s2_scene_5.tif
[11/14] Processing: s2_scene_6.tif
[12/14] Processing: s2_scene_7.tif
[13/14] Processing: s2_scene_8.tif
[14/14] Processing: s2_scene_9.tif

Saved summary to: /content/drive/MyDrive/WaterDetectionProject/phase1_summary.csv

Processing complete ✅
Total images processed: 14


,image,water_pixels,water_ratio
0,kuttanad_kerala_scene_1,387073,0.031558
1,kuttanad_kerala_scene_2,335062,0.027317
2,kuttanad_kerala_scene_3,343024,0.027966
3,kuttanad_s2_first_image,387073,0.031558
4,s2_scene_0,1166204,0.079293
